<a href="https://colab.research.google.com/github/MuhammadRivaldiAsyhari/AI-Driven-Transaction-Security/blob/main/AI_Driven_Transaction_Security.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Import and Load Dataset

link github : https://github.com/MuhammadRivaldiAsyhari/AI-Driven-Transaction-Security

In [ ]:
# menginstall library opendatasets agar dapat mengimport atau mengambil dataset langsung dari kaggle
!pip install pandas;
!pip install numpy;
!pip install seaborn;
!pip install matplotlib;
!pip install opendatasets;
!pip install -U scikit-learn;
!pip install xgboost;
!pip install statsmodels;

In [ ]:
# mengimport library yang nantinya akan di gunakan
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import opendatasets as od
plt.rcParams['figure.figsize'] = [10, 8]

import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import sklearn.metrics as metrics
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

import itertools
from collections import Counter

from statsmodels.stats.outliers_influence import variance_inflation_factor

In [ ]:
# menggunakan library od untuk mendownload dataset dari link kaggle atau dataset
# od.download(
#     "https://www.kaggle.com/datasets/chitwanmanchanda/fraudulent-transactions-data/data")

# pada kode di bawah untuk membaca file dataset yang telah di download
df = pd.read_csv("D:\clone github\AI-Driven-Transaction-Security\Fraud.csv")

# menampilkan data dari dataset untuk memastikan apakah sudah benar data yang terbaca
df.head()

In [ ]:
# melakukan cek informasi dari dataset seperti nama kolom dan tipe data dari setiap kolom
df.info()



```
Data Understanding
Dataset ini berisi 6.362.620 transaksi dengan 11 fitur, yang mencatat aktivitas keuangan selama 30 hari (744 jam) dalam sistem pembayaran digital.

*   step : maps a unit of time in the real world. In this case 1 step is 1 hour of time. Total steps 744 (30 days simulation).
*   type : CASH-IN, CASH-OUT, DEBIT, PAYMENT and TRANSFER.
*   amount : amount of the transaction in local currency.
*   nameOrig : customer who started the transaction
*   oldbalanceOrg : initial balance before the transaction
*   newbalanceOrig : new balance after the transaction
*   nameDest : customer who is the recipient of the transaction.
*   oldbalanceDest : initial balance recipient before the transaction. Note that there is not information for customers that start with M (Merchants).
*   newbalanceDest : new balance recipient after the transaction. Note that there is not information for customers that start with M (Merchants).
*   isFraud : This is the transactions made by the fraudulent agents inside the simulation. In this specific dataset the fraudulent behavior of the agents aims to profit by taking control or customers accounts and try to empty the funds by transferring to another account and then cashing out of the system.
*   isFlaggedFraud : The business model aims to control massive transfers from one account to another and flags illegal attempts. An illegal attempt in this dataset is an attempt to transfer more than 200.000 in a single transaction.
```





```
🎯 Penentuan Goals: AI-Driven Transaction Security
1️⃣ Tujuan Utama
Membangun sistem AI-driven security untuk mendeteksi transaksi mencurigakan menggunakan Fraud Detection (Supervised Learning) dan Anomaly Detection (Unsupervised Learning).

2️⃣ Sub-Goals (Tujuan Spesifik)
🔹 Fraud Detection (Supervised Learning)
Memprediksi apakah suatu transaksi adalah fraud (isFraud = 1) atau bukan.

Menggunakan model Random Forest, XGBoost, atau Neural Network.

Menangani imbalance data agar model tidak hanya mengenali transaksi normal.

🔹 Anomaly Detection (Unsupervised Learning)
Mengidentifikasi transaksi mencurigakan tanpa label fraud.

Menggunakan Isolation Forest, One-Class SVM, atau Autoencoder.

Membantu menemukan fraud baru yang belum pernah terlihat sebelumnya.

3️⃣ Metode Evaluasi
✅ Supervised Learning (Fraud Detection)

Precision & Recall → Mengurangi False Positive & False Negative.

F1-Score → Menyeimbangkan antara akurasi dan deteksi fraud yang efektif.

AUC-ROC → Menilai kemampuan model membedakan transaksi fraud dan non-fraud.

✅ Unsupervised Learning (Anomaly Detection)

Silhouette Score → Menilai kualitas klaster anomali.

Reconstruction Error (Autoencoder) → Menganalisis penyimpangan dari transaksi normal.

4️⃣ Manfaat dari Proyek Ini
✔ Meningkatkan keamanan transaksi dengan deteksi fraud lebih akurat.
✔ Mengurangi potensi kerugian finansial akibat transaksi mencurigakan.
✔ Dapat digunakan di dunia nyata untuk meningkatkan sistem keamanan perbankan digital.
```



# Data Cleaning

In [ ]:
# Cek apakah ada missing values
print(df.isnull().sum())

# Jika ada missing values, bisa dihapus atau diisi
df.dropna(inplace=True)  # Hapus baris dengan missing values (jika ada)


In [ ]:
# Cek duplikasi
print(f"Jumlah data duplikat: {df.duplicated().sum()}")

# Hapus duplikat jika ada
df.drop_duplicates(inplace=True)


In [ ]:
# Cek distribusi transaksi
plt.figure(figsize=(8,5))
sns.countplot(x=df["type"], palette="viridis")
plt.title("Distribusi Jenis Transaksi")
plt.show()

In [ ]:
# Cek distribusi jumlah transaksi
plt.figure(figsize=(8,5))
sns.boxplot(x=df["amount"])
plt.title("Distribusi Jumlah Transaksi")
plt.show()

In [ ]:
# Cek Balance Data Fraud vs Non-Fraud
# Karena biasanya data fraud sangat sedikit dibanding transaksi normal, ini akan jadi tantangan untuk model.
plt.figure(figsize=(6,4))
sns.countplot(x=df["isFraud"], palette="coolwarm")
plt.title("Distribusi Transaksi Fraud vs Non-Fraud")
plt.show()

fraud_percentage = (df["isFraud"].sum() / df.shape[0]) * 100
print(f"Persentase transaksi fraud: {fraud_percentage:.4f}%")


In [ ]:
df.to_csv("Fraud_Cleaned.csv", index=False)
print("Dataset cleaned dan disimpan sebagai 'Fraud_Cleaned.csv'")

# Data Manipulation

In [ ]:
# Load ulang dataset yang sudah di lakukan cleaning
df_cleaned = pd.read_csv('D:\clone github\AI-Driven-Transaction-Security\Fraud_Cleaned.csv')

df_cleaned.head()

In [ ]:
df_cleaned.info()

In [ ]:
# Encoding categorical data sebelum analisis korelasi
encoder = LabelEncoder()
df_cleaned['type'] = encoder.fit_transform(df_cleaned['type'])


In [ ]:
# Hapus kolom sender dan receiver sebelum analisis korelasi
df_corr = df_cleaned.drop(columns=['nameOrig', 'nameDest'])

# Visualisasi korelasi
plt.figure(figsize=(12, 8))
sns.heatmap(df_corr.corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title("Heatmap Korelasi")
plt.show()


In [ ]:
df_corr.head()

# EDA

In [ ]:
# 1 Cek statistik deskriptif
print(df_cleaned.describe())

In [ ]:
df_cleaned.head()

In [ ]:
# 2 Distribusi transaksi fraud vs non-fraud
plt.figure(figsize=(6, 4))
sns.countplot(x=df_cleaned["isFraud"], palette="coolwarm")
plt.title("Distribusi Transaksi Fraud vs Non-Fraud")
plt.xticks(ticks=[0, 1], labels=["Non-Fraud", "Fraud"])
plt.show()

In [ ]:
# 3 Visualisasi Distribusi Fitur Numerik
num_cols = ['amount', 'oldbalanceOrg', 'newbalanceOrig',
            'oldbalanceDest', 'newbalanceDest']

df_cleaned[num_cols].hist(figsize=(12, 8), bins=50, color='skyblue', edgecolor='black')
plt.suptitle("Distribusi Fitur Numerik", fontsize=16)
plt.show()



```
Insight dari Distribusi Fitur Numerik
Berdasarkan plot distribusi fitur numerik yang kamu hasilkan, ada beberapa hal yang bisa kita analisis:

*   Distribusi Data Sangat Tidak Merata (Skewed to the Left)

Hampir semua fitur numerik memiliki distribusi yang sangat condong ke kiri.

Ini menunjukkan bahwa sebagian besar nilai transaksi dan saldo berada dalam rentang yang kecil, sementara hanya sedikit transaksi yang memiliki nilai besar.

Dampak: Perlu dilakukan transformasi data (misalnya, log-transform) agar distribusinya lebih normal untuk algoritma yang sensitif terhadap distribusi data.

*   Banyaknya Nol pada Fitur Keuangan (Saldo & Transaksi)

Fitur seperti sender_old_balance, sender_new_balance, receiver_old_balance, dan receiver_new_balance memiliki banyak nilai yang mendekati nol.

Ini bisa menunjukkan bahwa banyak pengguna yang melakukan transaksi dengan saldo awal atau saldo akhir nol, yang mungkin berhubungan dengan fraud.

Dampak: Bisa menjadi fitur penting untuk model deteksi fraud, karena saldo yang sering kosong bisa menjadi pola tertentu.

*   Perbedaan Signifikan antara Pengirim dan Penerima

sender_balance_diff dan receiver_balance_diff juga menunjukkan pola yang mirip.

Kemungkinan besar ada perbedaan yang cukup besar antara pengirim dan penerima dalam hal saldo sebelum dan setelah transaksi.

Dampak: Bisa jadi fitur ini relevan untuk membedakan transaksi fraud dan non-fraud, karena fraudster mungkin sering mengosongkan saldo setelah transaksi.

*   Kebutuhan untuk Standarisasi & Scaling

Karena distribusi fitur numerik sangat tidak merata, langkah normalisasi seperti MinMaxScaler (yang sudah dilakukan sebelumnya) sangat diperlukan.

Alternatif lain seperti RobustScaler bisa dicoba jika dataset masih mengandung banyak outlier.
```





```
Analisis yang bisa kita ambil dari EDA ini:

*   Distribusi transaksi fraud vs non-fraud → Biasanya data fraud lebih sedikit dibanding non-fraud.
*   Distribusi fitur numerik → Kita bisa melihat apakah ada pola tertentu dalam transaksi fraud.
*   Korelasi antar fitur → Bisa membantu dalam feature selection sebelum modeling
```



In [ ]:
# Deteksi Outlier dengan Boxplot
plt.figure(figsize=(12, 6))
df_cleaned.boxplot(rot=90)
plt.title("Boxplot untuk Deteksi Outlier")
plt.show()

In [ ]:
# Deteksi Outlier dengan Z-Score
# z_scores = np.abs((df_cleaned - df_cleaned.mean()) / df_cleaned.std())
# outliers = (z_scores > 3).sum()
# print("Jumlah outlier per fitur:")
# print(outliers)

In [ ]:
# Number of fraud and legitimate transactions
fraud = len(df_cleaned[df_cleaned['isFraud'] == 1])
legit = len(df_cleaned[df_cleaned['isFraud'] == 0])

print("Number of Fraud transactions: ", fraud)
print("Number of Legit transactions: ", legit)

In [ ]:
df_cleaned.head()

In [ ]:
X = df_cleaned[df_cleaned['nameDest'].str.contains('M')]
X.head()

In [ ]:
# checking correlation
corr = df_cleaned.corr(numeric_only=True)
corr

In [ ]:
# plotting correlation using heatmap
sns.heatmap(data=corr, annot=True)

In [ ]:
# plotting bar chart for legit & fraud transaction
plt.figure(figsize=(6,6))
labels = ["Legit", "Fraud"]
count_classes = df_cleaned.value_counts(df_cleaned['isFraud'], sort= True)
count_classes.plot(kind = "bar", rot = 0)
plt.title("Visualization of Labels")
plt.ylabel("Count")
plt.xticks(range(2), labels)
plt.show()

# Feature Engineering

In [ ]:
# create a copy of the original dataframe
df_new = df_cleaned.copy()
df_new.head()

## label encoding

In [ ]:
# check object datatypes
objList = df_new.select_dtypes(include = "object").columns
print(objList)

In [ ]:
# encode the objects
le = LabelEncoder()

for f in objList:
    df_new[f] = le.fit_transform(df_new[f].astype(str))

print(df_new.info())

In [ ]:
df_new.head()

##Multicollinearity

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
# function to find the variation inflation factor
def cal_vif(df):
    vif = pd.DataFrame()
    vif['variables'] = df.columns
    vif['VIF'] = [variance_inflation_factor(df.values,i) for i in range(df.shape[1])]
    return vif

cal_vif(df_new)

- We can see that oldbalanceOrg and newbalanceOrig have too high VIF thus they are highly correlated. Similarly oldbalanceDest and newbalanceDest. Also nameDest is connected to nameOrig.

- Thus combine these pairs of collinear attributes and drop the individual ones.

In [ ]:
# creating new features to capture the change in balances and transaction paths
df_new['Actual_amount_orig'] = df_new.apply(lambda x: x['oldbalanceOrg'] - x['newbalanceOrig'],axis=1)
df_new['Actual_amount_dest'] = df_new.apply(lambda x: x['oldbalanceDest'] - x['newbalanceDest'],axis=1)
df_new['TransactionPath'] = df_new.apply(lambda x: x['nameOrig'] + x['nameDest'],axis=1)

#Dropping columns
df_new = df_new.drop(['oldbalanceOrg','newbalanceOrig','oldbalanceDest','newbalanceDest','step','nameOrig','nameDest'],axis=1)

cal_vif(df_new)

In [ ]:
# new correlation heatmap
corr=df_new.corr()

sns.heatmap(corr,annot=True)

How did you select variables to be included in the model?
   
- Using the VIF values and correlation heatmap. We just need to check if there are any two attributes highly correlated to each other and then drop the one which is less correlated to the isFraud Attribute.

# Modelling Machine Learning

## Scalling Data

In [ ]:
# scale the dataset
scaler = StandardScaler()
df_new["NormalizedAmount"] = scaler.fit_transform(df_new["amount"].values.reshape(-1, 1))
df_new.drop(["amount"], inplace= True, axis= 1)

Y = df_new["isFraud"]
X = df_new.drop(["isFraud"], axis= 1)

## Splitting Data

In [ ]:
# split the dataset for training and testing
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size= 0.3, random_state= 42)

print("Shape of X_train: ", X_train.shape)
print("Shape of X_test: ", X_test.shape)

## Model Training

In [ ]:
# Decision Tree
dtc = DecisionTreeClassifier()
dtc.fit(X_train, Y_train)

Y_pred_dt = dtc.predict(X_test)
dtc_score = dtc.score(X_test, Y_test) * 100

In [ ]:
# Random Forest
rfc = RandomForestClassifier(n_estimators= 100)
rfc.fit(X_train, Y_train)

Y_pred_rf = rfc.predict(X_test)
rfc_score = rfc.score(X_test, Y_test) * 100

In [ ]:
# XGBoost
xgb = XGBClassifier()
xgb.fit(X_train, Y_train)

Y_pred_xgb = xgb.predict(X_test)
xgb_score = xgb.score(X_test, Y_test) * 100

In [ ]:
print("Decision Tree Score: ", dtc_score)
print("Random Forest Score: ", rfc_score)
print("XGBoost Score      : ", xgb_score)

In [ ]:
# key terms of Confusion Matrix - DT

print("TP,FP,TN,FN - Decision Tree")
tn, fp, fn, tp = confusion_matrix(Y_test, Y_pred_dt).ravel()
print(f'True Positives: {tp}')
print(f'False Positives: {fp}')
print(f'True Negatives: {tn}')
print(f'False Negatives: {fn}')

print("----------------------------------------------------------------------------------------")

# key terms of Confusion Matrix - RF

print("TP,FP,TN,FN - Random Forest")
tn, fp, fn, tp = confusion_matrix(Y_test, Y_pred_rf).ravel()
print(f'True Positives: {tp}')
print(f'False Positives: {fp}')
print(f'True Negatives: {tn}')
print(f'False Negatives: {fn}')

print("----------------------------------------------------------------------------------------")

# key terms of Confusion Matrix - XGB

print("TP,FP,TN,FN - XGBoost")
tn, fp, fn, tp = confusion_matrix(Y_test, Y_pred_xgb).ravel()
print(f'True Positives: {tp}')
print(f'False Positives: {fp}')
print(f'True Negatives: {tn}')
print(f'False Negatives: {fn}')

In [ ]:
# confusion matrix - DT

confusion_matrix_dt = confusion_matrix(Y_test, Y_pred_dt.round())
print("Confusion Matrix - Decision Tree")
print(confusion_matrix_dt,)

print("----------------------------------------------------------------------------------------")

# confusion matrix - RF

confusion_matrix_rf = confusion_matrix(Y_test, Y_pred_rf.round())
print("Confusion Matrix - Random Forest")
print(confusion_matrix_rf)

print("----------------------------------------------------------------------------------------")

# confusion matrix - XGB

confusion_matrix_xgb = confusion_matrix(Y_test, Y_pred_xgb.round())
print('Confusion Matrix - XGBoost')
print(confusion_matrix_xgb)

In [ ]:
# classification report - DT

classification_report_dt = classification_report(Y_test, Y_pred_dt)
print("Classification Report - Decision Tree")
print(classification_report_dt)

print("----------------------------------------------------------------------------------------")

# classification report - RF

classification_report_rf = classification_report(Y_test, Y_pred_rf)
print("Classification Report - Random Forest")
print(classification_report_rf)

print("----------------------------------------------------------------------------------------")

# classification report - XGB

classification_report_xgb = classification_report(Y_test, Y_pred_xgb)
print("Classification Report - XGBoost")
print(classification_report_xgb)

In [ ]:
# visualising confusion matrix - DT

disp = ConfusionMatrixDisplay(confusion_matrix=confusion_matrix_dt)
disp.plot()
plt.title('Confusion Matrix - DT')
plt.show()

In [ ]:
# visualising confusion matrix - RF
disp = ConfusionMatrixDisplay(confusion_matrix=confusion_matrix_rf)
disp.plot()
plt.title('Confusion Matrix - RF')
plt.show()

In [ ]:
# visualising confusion matrix - XGB
disp = ConfusionMatrixDisplay(confusion_matrix=confusion_matrix_xgb)
disp.plot()
plt.title('Confusion Matrix - XGB')
plt.show()

In [ ]:
# AUC ROC - DT
# calculate the fpr and tpr for all thresholds of the classification

fpr, tpr, threshold = metrics.roc_curve(Y_test, Y_pred_dt)
roc_auc = metrics.auc(fpr, tpr)

plt.title('ROC - DT')
plt.plot(fpr, tpr, 'b', label = 'AUC = %0.2f' % roc_auc)
plt.legend(loc = 'lower right')
plt.plot([0, 1], [0, 1],'r--')
plt.xlim([0, 1])
plt.ylim([0, 1])
plt.ylabel('True Positive Rate')
plt.xlabel('False Positive Rate')
plt.show()

In [ ]:
# AUC ROC - RF
# calculate the fpr and tpr for all thresholds of the classification

fpr, tpr, threshold = metrics.roc_curve(Y_test, Y_pred_rf)
roc_auc = metrics.auc(fpr, tpr)

plt.title('ROC - RF')
plt.plot(fpr, tpr, 'b', label = 'AUC = %0.2f' % roc_auc)
plt.legend(loc = 'lower right')
plt.plot([0, 1], [0, 1],'r--')
plt.xlim([0, 1])
plt.ylim([0, 1])
plt.ylabel('True Positive Rate')
plt.xlabel('False Positive Rate')
plt.show()

In [ ]:
# AUC ROC - XGB
# calculate the fpr and tpr for all thresholds of the classification

fpr, tpr, threshold = metrics.roc_curve(Y_test, Y_pred_xgb)
roc_auc = metrics.auc(fpr, tpr)

plt.title('ROC - RF')
plt.plot(fpr, tpr, 'b', label = 'AUC = %0.2f' % roc_auc)
plt.legend(loc = 'lower right')
plt.plot([0, 1], [0, 1],'r--')
plt.xlim([0, 1])
plt.ylim([0, 1])
plt.ylabel('True Positive Rate')
plt.xlabel('False Positive Rate')
plt.show()

# Conclusion:

We have seen that Accuracy of Random Forest, Decision Tree and XGBoost is equal, although the precision of XGBoost is more. In a fraud detection model, Precision is highly important because rather than predicting normal transactions correctly we want to know about the Fraud transactions to be predicted correctly and Legit to be left off. If either of the 2 reasons are not fulfiiled we may catch the innocent and leave the culprit.

- This is also one of the reason why Ensemble techniques are used unstead of other algorithms.

- Also the reason I have chosen this model is because of highly unbalanced dataset (Legit: Fraud :: 6354407:8213). XGBoost builds a predictive model by combining the predictions of multiple individual models, often decision trees, in an iterative manner.


- Models like Bagging, ANN, and Logistic Regression may give good accuracy but they won't give good precision and recall values.

<b/> What are the key factors that predict fraudulent customer?

- Transaction amount.
- Changes in balances.
- Transaction type.
- Whether the source of payment request is secured or not?
- Is the receiver organization is legitimate or not?

<b/> Do These Factors Make Sense? If Yes, How? If Not, How Not?

Yes, these factors make sense:
- Transaction Amount: Large or unusual amounts can indicate fraud.
- Balance Changes: Significant changes in balances might signal unauthorized transactions.
- Transaction Type: Certain transaction types might be more prone to fraud (e.g., international transfers).

<b/> What Kind of Prevention Should Be Adopted While Company Update Its Infrastructure?
- Real-Time Monitoring
- Multi-Factor Authentication (MFA)
- User Behavior Analytics
- Regular Audits
- Data Encryption

<b/> Assuming These Actions Have Been Implemented, How Would You Determine If They Work?

To determine if the actions are effective:

- Monitor Fraud Rates: Track the fraud rates before and after implementation.
- User Feedback: Gather feedback from users on the new security measures.
- Performance Metrics: Continuously monitor model performance metrics (accuracy, precision, recall, ROC AUC).
- Regular Reviews: Conduct regular reviews of security incidents to assess improvements.

In [ ]:
print('done')